# 유동성: 식과 시뮬레이션 (13회차)

자격 비율 $q$와 연간 순매수 한도가 체결률을 얼마나 줄이는지, 식 (13.1)·(13.2)와 `s13_liquidity_sim.py`의 시뮬레이션을 같은 그림에 놓는다. 모형의 숫자는 가정($N=5{,}000$, $M=20$, $B=5\%$, 1회 2천만 원)에서 나온 것이고 실제 시장으로 보정하지 않았다.

In [ ]:
import sys, pathlib
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager
LABS = pathlib.Path.cwd().parent / "labs"
for f in ["Noto Sans CJK KR", "AppleGothic", "Malgun Gothic", "NanumGothic"]:
    if any(f == x.name for x in font_manager.fontManager.ttflist):
        plt.rcParams["font.family"] = f; break
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110,
                     "axes.unicode_minus": False})
ACC, GREY = "#2F5D8C", "#8E8C87"
sys.path.insert(0, str(LABS / 'part1'))
import s13_liquidity_sim as s13
print(s13.N, s13.M, s13.B, s13.LOT, s13.LIMIT)

## 1. 식 (13.1): 한도가 없을 때

하루에 적어도 한 명의 자격·의향 있는 후보를 만날 확률 $p=1-(1-qB)^M$, 21번(0–20일) 안에 체결될 확률 $1-(1-p)^{21}$.

In [ ]:
qs = np.array([1.0, 0.5, 0.2, 0.1, 0.05])
p = 1 - (1 - qs * s13.B) ** s13.M
fill_formula = 1 - (1 - p) ** 21
cap = np.minimum(1, (s13.LIMIT // s13.LOT) * qs * s13.N / (s13.SELLERS_PER_DAY * s13.PERIODS))
for q, a, b in zip(qs, fill_formula, cap):
    print(f"q={q:4.2f}  식(13.1) {a:6.1%}   한도 상한(13.2) {b:6.1%}")

## 2. 시뮬레이션

`s13.simulate`를 그대로 부른다(시드 고정, 1–2분). 한도가 있는 경우는 식 (13.2)의 상한에 붙는다.

In [ ]:
sim = [(q, *s13.simulate(q, False), *s13.simulate(q, True)) for q in qs]
for q, f0, w0, f1, w1 in sim:
    print(f"q={q:4.2f}  한도 없음 {f0:6.1%} (평균 대기 {w0:4.2f}일)   1억 한도 {f1:6.1%} (평균 대기 {w1:4.2f}일)")

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 3.8))
ax.plot(qs, fill_formula * 100, "--", color=GREY, label="식 (13.1) 한도 없음")
ax.plot(qs, cap * 100, ":", color=ACC, label="식 (13.2) 한도 상한")
ax.plot(qs, [r[1] * 100 for r in sim], "o-", color=GREY, label="시뮬레이션 한도 없음")
ax.plot(qs, [r[3] * 100 for r in sim], "s-", color=ACC, label="시뮬레이션 1억 한도")
ax.set_xscale("log"); ax.set_xticks(qs); ax.set_xticklabels([str(q) for q in qs])
ax.set_xlabel("자격 비율 q (로그)"); ax.set_ylabel("1년 체결률 (%)"); ax.legend(fontsize=8, frameon=False)
plt.tight_layout()

## 3. 한도를 바꾸면

상한 식에서 한도만 바꿔 본다(문제집 13.2). 한도가 2억이면 $q=0.2$의 상한이 1에 닿아, 그때부터는 탐색 마찰이 체결률을 정한다.

In [ ]:
for limit in [1e8, 2e8, 3e8]:
    lots = int(limit // s13.LOT)
    c = np.minimum(1, lots * qs * s13.N / (s13.SELLERS_PER_DAY * s13.PERIODS))
    print(f"한도 {limit/1e8:.0f}억 (1인 연 {lots}건): " + ", ".join(f"q={q}: {x:.0%}" for q, x in zip(qs, c)))

출처: `labs/part1/s13_liquidity_sim.py`; Duffie, Gârleanu and Pedersen (2005); 금융위원회 시행령 입법예고(2026-10-01)의 장외거래소별 일반투자자 연간 순매수 1억 원.